In [19]:
import pandas as pd
import numpy as np
import random

# ============================================================
# 1. REPRODUCIBILITY
# ============================================================

np.random.seed(42)
random.seed(42)

n = 25000


# ============================================================
# 2. SUPPLIER MASTER DATA
# ============================================================

suppliers = pd.DataFrame({
    "supplier_id": [f"SUP{i:03d}" for i in range(1, 26)],

    "supplier_name": [
        "Apex Industrial",
        "Nova Components",
        "Titan Metals",
        "Precision Parts",
        "Global Fasteners",
        "Vertex Manufacturing",
        "Prime Plastics",
        "Atlas Steel",
        "Summit Electronics",
        "Orion Packaging",
        "Delta Components",
        "Metro Industrial",
        "Evergreen Materials",
        "Fusion Manufacturing",
        "Pioneer Metals",
        "Sterling Components",
        "Horizon Plastics",
        "Crown Industrial",
        "Vector Electronics",
        "Liberty Packaging",
        "Pacific Metals",
        "Central Fasteners",
        "Elite Components",
        "Dynamic Materials",
        "United Manufacturing"
    ]
})

# Assign each supplier a region
suppliers["region"] = np.random.choice(
    ["North America", "Europe", "Asia"],
    size=25,
    p=[0.45, 0.20, 0.35]
)


# ============================================================
# 3. BUSINESS DIMENSIONS
# ============================================================

categories = [
    "Machined Components",
    "Raw Metals",
    "Fasteners",
    "Electronics",
    "Plastics",
    "Packaging"
]

category_probs = [
    0.22,
    0.20,
    0.17,
    0.15,
    0.14,
    0.12
]

plants = [
    "Boston Plant",
    "Chicago Plant",
    "Dallas Plant"
]

buyers = [
    "Buyer A",
    "Buyer B",
    "Buyer C",
    "Buyer D"
]


# ============================================================
# 4. SUPPLIER BEHAVIOR
# ============================================================

# Most suppliers generally deliver slightly early
supplier_delay = np.random.uniform(
    -4,
    -1,
    25
)

# Supplier-specific quality behavior
supplier_defect = np.random.uniform(
    0.002,
    0.035,
    25
)

# Supplier-specific pricing behavior
supplier_price_bias = np.random.uniform(
    -0.03,
    0.08,
    25
)


# ------------------------------------------------------------
# Intentionally create several business problems
# ------------------------------------------------------------

# Titan Metals: delivery problems
supplier_delay[2] = 3

# Atlas Steel: severe delivery problems
supplier_delay[7] = 5

# Global Fasteners: quality problems
supplier_defect[4] = 0.065

# Pioneer Metals: quality problems
supplier_defect[14] = 0.055

# Summit Electronics: unfavorable pricing
supplier_price_bias[8] = 0.14


# ============================================================
# 5. SUPPLIER SPEND / ORDER DISTRIBUTION
# ============================================================

supplier_probs = np.array([
    0.09,
    0.08,
    0.07,
    0.06,
    0.06,
    0.05,
    0.05,
    0.05,
    0.05,
    0.04,
    0.04,
    0.04,
    0.04,
    0.035,
    0.035,
    0.03,
    0.03,
    0.03,
    0.025,
    0.025,
    0.02,
    0.02,
    0.02,
    0.015,
    0.015
])

# Automatically normalize probabilities
supplier_probs = supplier_probs / supplier_probs.sum()


# ============================================================
# 6. GENERATE PROCUREMENT TRANSACTIONS
# ============================================================

records = []

start_date = pd.Timestamp("2024-01-01")
end_date = pd.Timestamp("2025-12-31")

date_range_days = (
    end_date - start_date
).days


for i in range(n):

    # --------------------------------------------------------
    # Select supplier
    # --------------------------------------------------------

    supplier_idx = np.random.choice(
        range(25),
        p=supplier_probs
    )

    supplier = suppliers.iloc[supplier_idx]


    # --------------------------------------------------------
    # Order date
    # --------------------------------------------------------

    order_date = start_date + pd.to_timedelta(
        np.random.randint(
            0,
            date_range_days + 1
        ),
        unit="D"
    )


    # --------------------------------------------------------
    # Category
    # --------------------------------------------------------

    category = np.random.choice(
        categories,
        p=category_probs
    )


    # --------------------------------------------------------
    # Quantity
    # --------------------------------------------------------

    quantity = np.random.randint(
        20,
        1500
    )


    # --------------------------------------------------------
    # Pricing
    # --------------------------------------------------------

    standard_price = round(
        np.random.uniform(
            5,
            250
        ),
        2
    )

    price_noise = np.random.normal(
        supplier_price_bias[supplier_idx],
        0.03
    )

    actual_price = round(
        max(
            1,
            standard_price * (1 + price_noise)
        ),
        2
    )


    # --------------------------------------------------------
    # Promised delivery
    # --------------------------------------------------------

    promised_lead_time = np.random.randint(
        10,
        45
    )

    promised_date = order_date + pd.Timedelta(
        days=promised_lead_time
    )


    # --------------------------------------------------------
    # Actual delivery performance
    # --------------------------------------------------------

    delivery_variance = round(
        np.random.normal(
            supplier_delay[supplier_idx],
            2.5
        )
    )

    # Prevent delivery before purchase order date
    minimum_variance = -(
        promised_lead_time - 1
    )

    delivery_variance = max(
        delivery_variance,
        minimum_variance
    )

    actual_delivery_date = (
        promised_date
        + pd.Timedelta(
            days=delivery_variance
        )
    )


    # --------------------------------------------------------
    # Quantity fulfillment
    # --------------------------------------------------------

    # 90% of purchase orders arrive completely in full
    if np.random.random() < 0.90:

        quantity_received = quantity

    else:

        shortage_pct = np.random.uniform(
            0.01,
            0.05
        )

        quantity_received = max(
            0,
            round(
                quantity
                * (1 - shortage_pct)
            )
        )


    # --------------------------------------------------------
    # Quality
    # --------------------------------------------------------

    defective_units = np.random.binomial(
        quantity_received,
        supplier_defect[supplier_idx]
    )


    # --------------------------------------------------------
    # Order value
    # --------------------------------------------------------

    order_value = round(
        quantity * actual_price,
        2
    )


    # --------------------------------------------------------
    # Add transaction
    # --------------------------------------------------------

    records.append({

        "po_number":
            f"PO{100000 + i}",

        "order_date":
            order_date,

        "supplier_id":
            supplier["supplier_id"],

        "supplier_name":
            supplier["supplier_name"],

        "supplier_region":
            supplier["region"],

        "category":
            category,

        "material_id":
            f"MAT{np.random.randint(1000, 9999)}",

        "quantity_ordered":
            quantity,

        "quantity_received":
            quantity_received,

        "standard_unit_price":
            standard_price,

        "actual_unit_price":
            actual_price,

        "order_value":
            order_value,

        "promised_delivery_date":
            promised_date,

        "actual_delivery_date":
            actual_delivery_date,

        "defective_units":
            defective_units,

        "payment_terms":
            np.random.choice(
                [
                    "Net 30",
                    "Net 45",
                    "Net 60"
                ]
            ),

        "buyer":
            np.random.choice(
                buyers
            ),

        "plant":
            np.random.choice(
                plants
            )
    })


# ============================================================
# 7. CREATE DATAFRAME
# ============================================================

df = pd.DataFrame(records)


# ============================================================
# 8. BASIC VALIDATION
# ============================================================

print("Dataset generated successfully.")
print("Number of records:", len(records))
print("DataFrame shape:", df.shape)

df.head()

Dataset generated successfully.
Number of records: 25000
DataFrame shape: (25000, 18)


,po_number,order_date,supplier_id,supplier_name,supplier_region,category,material_id,quantity_ordered,quantity_received,standard_unit_price,actual_unit_price,order_value,promised_delivery_date,actual_delivery_date,defective_units,payment_terms,buyer,plant
0,PO100000,2024-08-28,SUP001,Apex Industrial,North America,Raw Metals,MAT4157,115,115,175.40,185.77,21363.55,2024-09-21,2024-09-17,2,Net 45,Buyer B,Boston Plant
1,PO100001,2025-10-07,SUP005,Global Fasteners,North America,Fasteners,MAT6864,1383,1383,223.68,229.03,316748.49,2025-11-13,2025-11-10,84,Net 60,Buyer C,Chicago Plant
2,PO100002,2025-10-10,SUP009,Summit Electronics,Europe,Fasteners,MAT6232,1268,1268,34.37,39.65,50276.20,2025-11-20,2025-11-18,5,Net 45,Buyer A,Chicago Plant
3,PO100003,2025-05-24,SUP023,Elite Components,North America,Electronics,MAT6249,1430,1430,78.72,80.20,114686.00,2025-06-25,2025-06-29,6,Net 30,Buyer D,Chicago Plant
4,PO100004,2025-02-19,SUP025,United Manufacturing,Europe,Electronics,MAT5282,1482,1482,191.60,192.67,285536.94,2025-03-24,2025-03-18,28,Net 45,Buyer D,Boston Plant


In [20]:
df.shape

(25000, 18)

In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 25000 entries, 0 to 24999
Data columns (total 18 columns):
 #   Column                  Non-Null Count  Dtype         
---  ------                  --------------  -----         
 0   po_number               25000 non-null  object        
 1   order_date              25000 non-null  datetime64[ns]
 2   supplier_id             25000 non-null  object        
 3   supplier_name           25000 non-null  object        
 4   supplier_region         25000 non-null  object        
 5   category                25000 non-null  object        
 6   material_id             25000 non-null  object        
 7   quantity_ordered        25000 non-null  int64         
 8   quantity_received       25000 non-null  int64         
 9   standard_unit_price     25000 non-null  float64       
 10  actual_unit_price       25000 non-null  float64       
 11  order_value             25000 non-null  float64       
 12  promised_delivery_date  25000 non-null  dateti

In [ ]:
df.head()

,po_number,order_date,supplier_id,supplier_name,supplier_region,category,material_id,quantity_ordered,quantity_received,standard_unit_price,actual_unit_price,order_value,promised_delivery_date,actual_delivery_date,defective_units,payment_terms,buyer,plant
0,PO100000,2025-07-17,SUP001,Apex Industrial,North America,Fasteners,MAT3693,504,504,66.08,72.33,36454.32,2025-08-08,2025-08-09,13,Net 45,Buyer B,Dallas Plant
1,PO100001,2025-04-17,SUP005,Global Fasteners,North America,Raw Metals,MAT1663,1268,1256,71.67,67.53,85628.04,2025-05-02,2025-04-26,86,Net 60,Buyer C,Chicago Plant
2,PO100002,2024-08-20,SUP003,Titan Metals,Asia,Raw Metals,MAT1854,461,455,240.80,235.40,108519.40,2024-09-15,2024-09-15,14,Net 30,Buyer D,Boston Plant
3,PO100003,2024-05-09,SUP012,Metro Industrial,Asia,Fasteners,MAT9004,749,745,246.48,264.54,198140.46,2024-05-29,2024-05-24,11,Net 45,Buyer B,Chicago Plant
4,PO100004,2025-08-31,SUP009,Summit Electronics,Europe,Machined Components,MAT2734,206,202,50.70,57.95,11937.70,2025-09-28,2025-09-30,0,Net 30,Buyer A,Dallas Plant


In [ ]:
df.to_csv("procurement_transactions.csv", index=False)

print("Dataset saved successfully.")

Dataset saved successfully.


In [21]:
df["actual_delivery_date"] = pd.to_datetime(df["actual_delivery_date"])
df["order_date"] = pd.to_datetime(df["order_date"])

(df["actual_delivery_date"] - df["order_date"]).dt.days.min()

1

In [10]:
import os

print(os.path.exists("procurement_transactions.csv"))

False


In [11]:
df.to_csv("/content/procurement_transactions.csv", index=False)

print("Saved:", "/content/procurement_transactions.csv")
print("Shape:", df.shape)

Saved: /content/procurement_transactions.csv
Shape: (25000, 18)


In [12]:
import os

print(os.path.exists("/content/procurement_transactions.csv"))

True


In [22]:
df.to_csv(
    "/content/procurement_transactions.csv",
    index=False
)

print("Final dataset saved successfully.")

Final dataset saved successfully.
